# Student-test preprocessing with frozen settings

Use training-derived radius, K, GPS flag threshold and alignment cost cap. No percentile, threshold, feature rule or model parameter is fitted on these ten student-test trips.

Predictor features are computed with **blank labels** and saved first. Reference routes are then read only to create separate evaluation metadata. The predictor file keeps labels blank; model prediction later selects only the shared predictor columns. This notebook does not tune or score any model.

In [1]:
%run 05_feature_helpers.ipynb
from pathlib import Path
import json
from pyproj import Transformer
from shapely.strtree import STRtree
import pyarrow as pa
import pyarrow.parquet as pq

contract = json.loads(Path("outputs/folds/contract.json").read_text())
settings = contract["candidate_settings"]
radius_m, k = settings["radius_m"], settings["K"]
threshold_m = contract["point_flag_threshold_m"]
cap_m = float(pd.read_csv("outputs/labels/summary.csv")["cost_cap_m"].iloc[0])
test_folder = Path("Student folder/student_test")
output = Path("outputs/student_test")
output.mkdir(parents=True, exist_ok=True)
offset = int(pd.read_csv(contract["fold_file"], usecols=["group_id"])["group_id"].max()) + 1
print("Frozen settings:", radius_m, "m | K", k, "| Flag threshold", threshold_m, "m | Cost cap", cap_m)

Frozen settings: 210 m | K 100 | Flag threshold 208.3570580176556 m | Cost cap 150.0


## Reference alignment for evaluation

Use the exact monotonic DP from notebook 03. Candidate selection and feature calculations never use its outputs.

In [2]:
def align_points(D, cap_m):
    n, r = D.shape
    parents = np.zeros((n, r), dtype=np.int32)
    previous = np.minimum(D[0], cap_m)

    for i in range(1, n):
        prefix = np.minimum.accumulate(previous)
        improved = np.r_[True, previous[1:] < prefix[:-1]]
        parents[i] = np.maximum.accumulate(np.where(improved, np.arange(r), 0))
        previous = np.minimum(D[i], cap_m) + prefix

    route_pos = np.zeros(n, dtype=np.int32)
    route_pos[-1] = previous.argmin()
    for i in range(n - 1, 0, -1):
        route_pos[i - 1] = parents[i, route_pos[i]]
    return route_pos

## Process the ten trips

Keep the nearest K segments by geometric distance, breaking ties by segment ID. Empty shortlists remain visible in the point metadata. Test group IDs start after the training group IDs.

The features file contains all candidate rows, including flagged points. Exclude flagged groups only when evaluating against reference labels, after prediction.

In [3]:
writer = None
point_tables = []
checks = []
try:
    for folder in sorted(test_folder.iterdir()):
        if not folder.is_dir():
            continue
        trip_id = folder.name
        nodes = pd.read_csv(folder / f"{trip_id}.nodes", sep=r"\s+", header=None, names=["lon", "lat"])
        arcs = pd.read_csv(folder / f"{trip_id}.arcs", sep=r"\s+", header=None).to_numpy()
        track = pd.read_csv(folder / f"{trip_id}.track", sep=r"\s+", header=None, names=["lon", "lat", "time"])
        zone = int((track["lon"].mean() + 180) // 6) + 1
        epsg = (32600 if track["lat"].mean() >= 0 else 32700) + zone
        projection = Transformer.from_crs("EPSG:4326", f"EPSG:{epsg}", always_xy=True)
        nodes_xy = np.column_stack(projection.transform(nodes["lon"], nodes["lat"]))
        gps_xy = np.column_stack(projection.transform(track["lon"], track["lat"]))
        segments = shapely.linestrings(nodes_xy[arcs])
        gps = shapely.points(gps_xy)
        p, s = STRtree(segments).query(gps, predicate="dwithin", distance=radius_m)
        candidates = pd.DataFrame({"point_id": p, "segment_id": s,
            "distance": shapely.distance(gps[p], segments[s])})
        candidates = candidates.sort_values(["point_id", "distance", "segment_id"]).reset_index(drop=True)
        candidates = candidates[candidates.groupby("point_id").cumcount() < k].copy()
        candidates["trip_id"] = trip_id
        candidates["group_id"] = candidates["point_id"] + offset
        candidates["label"] = pd.array([pd.NA] * len(candidates), dtype="Int8")
        candidates = candidates[["trip_id", "group_id", "point_id", "segment_id", "distance", "label"]]
        features = add_features(candidates, nodes_xy, arcs, gps_xy, track["time"].to_numpy())
        assert features["label"].isna().all() and features["group_id"].is_monotonic_increasing
        table = pa.Table.from_pandas(features, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(output / "features.parquet", table.schema, compression="zstd")
        writer.write_table(table, row_group_size=len(features))

        route = pd.read_csv(folder / f"{trip_id}.route", sep=r"\s+", header=None, names=["arc_id"])
        D = shapely.distance(gps[:, None], segments[route["arc_id"]][None, :])
        route_pos = align_points(D, cap_m)
        assert np.diff(route_pos).min(initial=0) >= 0
        points = track.copy()
        points.insert(0, "point_id", np.arange(len(track)))
        points.insert(0, "group_id", points["point_id"] + offset)
        points.insert(0, "trip_id", trip_id)
        points["route_pos"] = route_pos
        points["true_arc_id"] = route["arc_id"].to_numpy()[route_pos]
        points["dist_m"] = D[np.arange(len(track)), route_pos]
        points["bad_gps"] = points["dist_m"] > threshold_m
        points.loc[points["bad_gps"], "true_arc_id"] = -1
        counts = candidates.groupby("point_id").size()
        points["candidate_count"] = counts.reindex(points.index, fill_value=0)
        true_ids = points["true_arc_id"].to_numpy()[candidates["point_id"]]
        positive_points = candidates.loc[candidates["segment_id"].to_numpy() == true_ids, "point_id"]
        points["true_in_candidates"] = points["point_id"].isin(positive_points).astype("boolean")
        points.loc[points["bad_gps"], "true_in_candidates"] = pd.NA
        point_tables.append(points)
        checks.append({"trip_id": trip_id, "gps_points": len(track), "candidate_rows": len(features)})
        offset += len(track)
        del D
        print("Prepared student-test trip:", trip_id, flush=True)
finally:
    if writer is not None:
        writer.close()

points = pd.concat(point_tables, ignore_index=True)
points.to_csv(output / "points.csv", index=False)
pd.DataFrame(checks).to_csv(output / "summary.csv", index=False)
(output / "settings.json").write_text(json.dumps({"radius_m": radius_m, "K": k,
    "flag_threshold_m": threshold_m, "cost_cap_m": cap_m}, indent=2) + "\n")
assert len(checks) == 10 and points["group_id"].is_unique
assert pq.ParquetFile(output / "features.parquet").metadata.num_rows == points["candidate_count"].sum()
print("Prepared", len(checks), "student-test trips. Model scores are not computed here.")
contract["student_test_status"] = "features and separate evaluation metadata prepared with frozen training settings; model scores not yet computed"
Path("outputs/folds/contract.json").write_text(json.dumps(contract, indent=2) + "\n")

Prepared student-test trip: 00000000


Prepared student-test trip: 00000007


Prepared student-test trip: 00000009


Prepared student-test trip: 00000040


Prepared student-test trip: 00000052


Prepared student-test trip: 00000056


Prepared student-test trip: 00000065


Prepared student-test trip: 00000076


Prepared student-test trip: 00000077


Prepared student-test trip: 00000091


Prepared 10 student-test trips. Model scores are not computed here.
